# Sentiment Analysis & Reporting

In [1]:
from pathlib import Path
!pip -q install --no-cache-dir "numpy==1.26.4"

NLP_DIR = Path("NLP Data")

news_files = sorted(NLP_DIR.glob("news*.txt"))

news_texts = {}
for f in news_files:
    with open(f, "r", encoding="utf-8") as file:
        news_texts[f.stem] = file.read()

print(f"Loaded {len(news_texts)} files:")
for k in news_texts:
    print(k)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.24.1 requires torch==2.9.1, but you have torch 2.2.2 which is incompatible.
Loaded 5 files:
news1_1202
news2_1027
news3_1223
news4_1223
news5_1028


In [ ]:
import time
from pathlib import Path
import requests

API_KEY = "XXXXXX"

API_URL = "https://api.openai.com/v1/responses"

CLEAN_DIR = Path("NLP Data Cleaned")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
SYSTEM_PROMPT = (
    "You clean financial news text for downstream NLP. "
    "Remove boilerplate, navigation, ads, duplicate lines, disclaimers, copyright lines, "
    "and unrelated website UI fragments. Keep dates, numbers, and FX/macro content. "
    "Do not add facts. Return ONLY the cleaned text, no markdown, no commentary."
)

In [ ]:
def clean_with_gpt(raw_text: str, model: str = "gpt-4.1-mini", max_retries: int = 5) -> str:
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "input": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": raw_text[:120000]}  
        ],
        "text": {"format": {"type": "text"}}
    }

    for attempt in range(max_retries):
        r = requests.post(API_URL, headers=headers, json=payload, timeout=60)

        if r.status_code == 200:
            data = r.json()
            cleaned = data.get("output_text")
            if cleaned:
                return cleaned.strip()

            # Fallback: try to assemble from output items
            out = []
            for item in data.get("output", []):
                for c in item.get("content", []):
                    if c.get("type") in ("output_text", "text"):
                        out.append(c.get("text", ""))
            return "\n".join(out).strip()

        # Retry on rate limits / transient errors
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** attempt)
            continue

        # Hard fail otherwise
        raise RuntimeError(f"API error {r.status_code}: {r.text}")

    raise RuntimeError("Max retries exceeded.")

In [5]:
for doc_id, raw_text in news_texts.items():
    cleaned_text = clean_with_gpt(raw_text)

    out_file = CLEAN_DIR / f"{doc_id}.txt"
    out_file.write_text(cleaned_text, encoding="utf-8")

    print(f"✅ Saved cleaned file: {out_file.name}")

print(f"\nFinished. Cleaned files saved to: {CLEAN_DIR.resolve()}")

✅ Saved cleaned file: news1_1202.txt
✅ Saved cleaned file: news2_1027.txt
✅ Saved cleaned file: news3_1223.txt
✅ Saved cleaned file: news4_1223.txt
✅ Saved cleaned file: news5_1028.txt

Finished. Cleaned files saved to: /Users/nigelli/Desktop/Columbia MAFN/UBS FINAI/Code/NLP Data Cleaned


In [6]:
!pip -q install -U "transformers==4.41.2" "torch==2.2.2" "sentencepiece" "safetensors" --no-cache-dir

In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU only

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
    device=-1,
    truncation=True
)

/opt/anaconda3/envs/windpy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/windpy/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd


CLEAN_DIR = Path("NLP Data Cleaned")
clean_files = sorted(CLEAN_DIR.glob("news*.txt"))

def chunk_text_by_words(text, max_words=220):  
    words = text.split()
    for i in range(0, len(words), max_words):
        yield " ".join(words[i:i+max_words])

def finbert_doc_scores(text: str):
    probs = {"positive": 0.0, "negative": 0.0, "neutral": 0.0}
    n = 0
    for chunk in chunk_text_by_words(text, max_words=220):
        out = pipe(chunk)[0] 
        d = {x["label"].lower(): float(x["score"]) for x in out}
        for k in probs:
            probs[k] += d.get(k, 0.0)
        n += 1
    if n == 0:
        return {"positive": None, "negative": None, "neutral": None, "label": None, "confidence": None, "sent_score": None}

    for k in probs:
        probs[k] /= n

    # label + confidence + simple scalar (pos - neg)
    label = max(probs, key=probs.get)
    conf  = probs[label]
    sent_score = probs["positive"] - probs["negative"]

    return {**probs, "label": label, "confidence": conf, "sent_score": sent_score}

rows = []
for f in clean_files:
    doc_id = f.stem
    text = f.read_text(encoding="utf-8", errors="ignore").strip()
    sc = finbert_doc_scores(text)
    rows.append({"doc_id": doc_id, **sc, "n_chars": len(text)})

df_finbert = pd.DataFrame(rows).sort_values("doc_id").reset_index(drop=True)
df_finbert

,doc_id,positive,negative,neutral,label,confidence,sent_score,n_chars
0,news1_1202,0.102223,0.850414,0.047364,negative,0.850414,-0.748191,3263
1,news2_1027,0.499244,0.468448,0.032309,positive,0.499244,0.030796,4012
2,news3_1223,0.753010,0.068705,0.178285,positive,0.753010,0.684305,4422
3,news4_1223,0.028626,0.941993,0.029382,negative,0.941993,-0.913367,2390
4,news5_1028,0.035011,0.948774,0.016215,negative,0.948774,-0.913763,2368


In [9]:
display(df_finbert.sort_values("sent_score", ascending=False).head(10)[["doc_id","label","confidence","sent_score","positive","negative","neutral"]])
display(df_finbert.sort_values("sent_score", ascending=True).head(10)[["doc_id","label","confidence","sent_score","positive","negative","neutral"]])

,doc_id,label,confidence,sent_score,positive,negative,neutral
2,news3_1223,positive,0.753010,0.684305,0.753010,0.068705,0.178285
1,news2_1027,positive,0.499244,0.030796,0.499244,0.468448,0.032309
0,news1_1202,negative,0.850414,-0.748191,0.102223,0.850414,0.047364
3,news4_1223,negative,0.941993,-0.913367,0.028626,0.941993,0.029382
4,news5_1028,negative,0.948774,-0.913763,0.035011,0.948774,0.016215


,doc_id,label,confidence,sent_score,positive,negative,neutral
4,news5_1028,negative,0.948774,-0.913763,0.035011,0.948774,0.016215
3,news4_1223,negative,0.941993,-0.913367,0.028626,0.941993,0.029382
0,news1_1202,negative,0.850414,-0.748191,0.102223,0.850414,0.047364
1,news2_1027,positive,0.499244,0.030796,0.499244,0.468448,0.032309
2,news3_1223,positive,0.753010,0.684305,0.753010,0.068705,0.178285


# Comments for above:

1, 4, 5 is strongly negative 
3 is postiive 
2 is pretty netural

Note that there is very high confidence via sent_scores which are negative, and are unambiguously bearish

We can probably take a | sent_score | > 0.3 --> as a signal, and if lower ignore.

Sentiment score (sent_score) is a signed scalar measure of net tone for a document. defined as sent_score = P(positive) - P(Negative) - because we chunked it 

In [ ]:
import re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from dataclasses import dataclass
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

EPS = 1e-12
TENORS = ["1M", "3M", "6M", "1Y"]

# 0) Core helpers
def parse_mixed_date(x):
    dt = pd.to_datetime(x, errors="coerce")
    if not pd.isna(dt): return dt
    try:
        xf = float(x)
        if 30000 < xf < 50000:
            return pd.to_datetime("1899-12-30") + pd.to_timedelta(xf, unit="D")
    except: pass
    return pd.NaT

def _as_clean_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf,-np.inf], np.nan).dropna().sort_index()
    if not isinstance(s.index, pd.DatetimeIndex): raise TypeError("Need DatetimeIndex.")
    return s[~s.index.duplicated(keep="last")]

def dlog(s):   return np.log(pd.to_numeric(s, errors="coerce")).replace([np.inf,-np.inf], np.nan).diff()
def dlevel(s): return pd.to_numeric(s, errors="coerce").diff()

def rv_proxy(r: pd.Series, window: int) -> pd.Series:
    return (r.astype(float)**2).rolling(window, min_periods=window).mean()

def logrv_proxy(r: pd.Series, window: int) -> pd.Series:
    return np.log(rv_proxy(r, window).clip(EPS))

def qlike(y: pd.Series, yhat: pd.Series) -> float:
    a = y.astype(float).clip(EPS)
    f = yhat.reindex(a.index).astype(float).clip(EPS)
    return float(np.nanmean(np.log(f.values) + (a.values / f.values)))

def corr_logrv(y: pd.Series, yhat: pd.Series) -> float:
    a = np.log(y.astype(float).clip(EPS))
    f = np.log(yhat.reindex(a.index).astype(float).clip(EPS))
    df = pd.concat([a,f], axis=1).dropna()
    return float(df.corr().iloc[0,1]) if len(df) > 5 else np.nan


In [62]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

def parse_mixed_date(x):
    dt = pd.to_datetime(x, errors="coerce")
    if not pd.isna(dt):
        return dt
    try:
        xf = float(x)
        if 30000 < xf < 50000:
            return pd.to_datetime("1899-12-30") + pd.to_timedelta(xf, unit="D")
    except Exception:
        pass
    return pd.NaT

def load_panel0(path: str | Path):
    path = Path(path)
    suf = path.suffix.lower()

    if suf in (".xlsx", ".xls"):
        xls = pd.ExcelFile(path)
        blocks = []
        for sh in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sh)
            if df.shape[1] < 2:
                continue

            # find Date column
            date_col = None
            for c in df.columns:
                if str(c).strip().lower() in ("date", "dates", "time", "datetime"):
                    date_col = c
                    break
            if date_col is None:
                date_col = df.columns[0]

            df[date_col] = df[date_col].apply(parse_mixed_date)
            df = df.dropna(subset=[date_col]).rename(columns={date_col: "Date"}).set_index("Date").sort_index()
            df = df.loc[~df.index.duplicated(keep="last")]

            # coerce numeric
            for c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

            df = df.dropna(axis=1, how="all")
            if df.shape[1] > 0:
                blocks.append(df)

        if not blocks:
            raise ValueError(f"No usable sheets found in {path}")

        df = pd.concat(blocks, axis=1).sort_index()
        df = df.loc[~df.index.duplicated(keep="last")]

    elif suf == ".csv":
        df_raw = pd.read_csv(path, low_memory=False)
        if "Date" not in df_raw.columns:
            raise ValueError("CSV must have a 'Date' column.")
        df_raw["Date_parsed"] = df_raw["Date"].apply(parse_mixed_date)
        df = (df_raw.dropna(subset=["Date_parsed"])
                    .drop(columns=["Date"])
                    .rename(columns={"Date_parsed": "Date"})
                    .sort_values("Date")
                    .set_index("Date"))
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    assert isinstance(df.index, pd.DatetimeIndex)
    assert df.index.is_monotonic_increasing

    # taxonomy (same as your logic)
    SURFACE_COLS = [c for c in df.columns if re.match(r"^(CNY|CNH)_(1M|3M|6M|1Y)_(ATM|25DRR|25DBF|10DRR|10DBF)$", str(c))]
    SPOT_COLS = [c for c in ["CNY_SPOT","CNH_SPOT","CNY_CNY_PBOC_FIXING","CNH_CNH_PBOC_FIXING","FEAT_CNY_Spot_Fix_Dev"] if c in df.columns]
    DRIVER_LEVEL_COLS = [c for c in df.columns if str(c).startswith(("DRV_","POL_","CN_","US_","OTH_"))]
    RETURN_COLS = [c for c in df.columns if str(c).startswith("RET_")]
    CHANGE_COLS = [c for c in df.columns if str(c).startswith("CHG_")]

    panel = df[SURFACE_COLS + SPOT_COLS + DRIVER_LEVEL_COLS + RETURN_COLS + CHANGE_COLS].copy()
    panel = panel.dropna(subset=SURFACE_COLS, how="all")  # keep only days with some surface
    panel = panel.apply(pd.to_numeric, errors="coerce").sort_index()

    TAX = dict(
        SURFACE_COLS=SURFACE_COLS, SPOT_COLS=SPOT_COLS,
        DRIVER_LEVEL_COLS=DRIVER_LEVEL_COLS, RETURN_COLS=RETURN_COLS, CHANGE_COLS=CHANGE_COLS
    )
    return panel.copy(), TAX

In [51]:
TENORS = ["1M","3M","6M","1Y"]
def surface_block_cols(df, mkt, smile):
    cols = []
    for t in TENORS:
        cols += [c for c in df.columns if re.match(rf"^{mkt}_{t}_{smile}$", str(c))]
    return cols

def first_valid_date(df, cols):
    if not cols: return None
    ok = df[cols].notna().any(axis=1)
    return ok.index[ok.values][0] if ok.any() else None


In [52]:
def surface_block_cols(df: pd.DataFrame, mkt: str, smile: str):
    cols = []
    for t in TENORS:
        cols += [c for c in df.columns if re.match(rf"^{mkt}_{t}_{smile}$", str(c))]
    return cols

def first_valid_date(df: pd.DataFrame, cols: list[str]):
    if not cols:
        return None
    ok = df[cols].notna().any(axis=1)
    return ok.index[ok.values][0] if ok.any() else None

def make_pca_input(df: pd.DataFrame, cols: list[str], mode: str):
    if not cols:
        return pd.DataFrame(index=df.index)
    out = {c: (dlog(df[c]) if mode == "dlog" else dlevel(df[c])) for c in cols}
    return pd.DataFrame(out, index=df.index)

@dataclass
class PCAModel:
    cols: list[str]
    mode: str
    scaler: StandardScaler
    pca: PCA
    evr: np.ndarray
    loadings: pd.DataFrame

def fit_pca_block(df_is: pd.DataFrame, cols: list[str], mode: str, n_components: int) -> PCAModel:
    if not cols:
        raise ValueError("Empty PCA block.")
    X = make_pca_input(df_is, cols, mode=mode).dropna(how="any")
    if X.shape[0] < max(80, 10 * n_components):
        raise ValueError(f"Too few rows for PCA: {X.shape[0]}")

    sc = StandardScaler()
    Xz = sc.fit_transform(X.values)
    pca = PCA(n_components=n_components, random_state=0).fit(Xz)

    pc_cols = [f"PC{i+1}" for i in range(n_components)]
    load = pd.DataFrame(pca.components_.T, index=X.columns, columns=pc_cols)

    return PCAModel(list(X.columns), mode, sc, pca, pca.explained_variance_ratio_, load)

def apply_pca_block(df: pd.DataFrame, mdl: PCAModel, prefix: str) -> pd.DataFrame:
    X = make_pca_input(df, mdl.cols, mode=mdl.mode).sort_index().ffill()
    X = X.fillna(X.median(numeric_only=True))
    Xz = mdl.scaler.transform(X.values)
    scores = mdl.pca.transform(Xz)
    cols = [f"{prefix}_PC{i+1}" for i in range(mdl.pca.n_components_)]
    return pd.DataFrame(scores, index=X.index, columns=cols)

def build_factors(panel0: pd.DataFrame, is_end: pd.Timestamp):
    panel_is = panel0.loc[:is_end].copy()
    pf = panel0.copy()

    # blocks (CNY)
    ATM_CNY  = surface_block_cols(panel0, "CNY", "ATM")
    RR25_CNY = surface_block_cols(panel0, "CNY", "25DRR")
    BF25_CNY = surface_block_cols(panel0, "CNY", "25DBF")
    RR10_CNY = surface_block_cols(panel0, "CNY", "10DRR")
    BF10_CNY = surface_block_cols(panel0, "CNY", "10DBF")

    # blocks (CNH)
    ATM_CNH  = surface_block_cols(panel0, "CNH", "ATM")
    RR25_CNH = surface_block_cols(panel0, "CNH", "25DRR")
    BF25_CNH = surface_block_cols(panel0, "CNH", "25DBF")
    RR10_CNH = surface_block_cols(panel0, "CNH", "10DRR")
    BF10_CNH = surface_block_cols(panel0, "CNH", "10DBF")

    CNH_START = first_valid_date(panel0, ATM_CNH + RR25_CNH + BF25_CNH + RR10_CNH + BF10_CNH)

    # --- CNY PCA (fit on IS only) ---
    pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is, ATM_CNY,  "dlog",   2), "CNY_ATM"))
    pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is, RR25_CNY, "dlevel", 1), "CNY_25RR"))
    pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is, BF25_CNY, "dlevel", 1), "CNY_25BF"))

    if len(RR10_CNY) >= 2:
        pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is, RR10_CNY, "dlevel", 1), "CNY_10RR"))
    if len(BF10_CNY) >= 2:
        pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is, BF10_CNY, "dlevel", 1), "CNY_10BF"))

    # --- CNH PCA (fit on IS but only after CNH exists) ---
    if CNH_START is not None:
        panel_is_cnh = panel_is.loc[panel_is.index >= CNH_START].copy()
        if len(ATM_CNH) >= 2:
            pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is_cnh, ATM_CNH, "dlog", 2), "CNH_ATM"))
        if len(RR25_CNH) >= 2:
            pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is_cnh, RR25_CNH, "dlevel", 1), "CNH_25RR"))
        if len(BF25_CNH) >= 2:
            pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is_cnh, BF25_CNH, "dlevel", 1), "CNH_25BF"))
        if len(RR10_CNH) >= 2:
            pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is_cnh, RR10_CNH, "dlevel", 1), "CNH_10RR"))
        if len(BF10_CNH) >= 2:
            pf = pf.join(apply_pca_block(panel0, fit_pca_block(panel_is_cnh, BF10_CNH, "dlevel", 1), "CNH_10BF"))

    FACTORS = [c for c in pf.columns if re.match(r"^(CNY|CNH)_(ATM|25RR|25BF|10RR|10BF)_PC\d+$", str(c))]
    return pf, FACTORS, CNH_START

In [53]:
def make_ar1_residuals(panel_factors: pd.DataFrame, factor_cols: list[str], is_end: pd.Timestamp):
    resid_map = {}
    for c in factor_cols:
        s = panel_factors[c].dropna().astype(float)
        s_is = s.loc[:is_end]
        if len(s_is) < 300:
            continue

        y = s_is.iloc[1:].values
        X = sm.add_constant(s_is.iloc[:-1].values)
        fit = sm.OLS(y, X).fit()

        c0, phi = float(fit.params[0]), float(fit.params[1])
        e = (s - (c0 + phi * s.shift(1))).dropna()
        resid_map[c] = e.rename(f"{c}_resid_AR1")
    return resid_map

def standardise_resid(resid: pd.Series, is_end: pd.Timestamp):
    r = _as_clean_series(resid).astype(float)
    sd = float(r.loc[:is_end].std(ddof=0))
    if (not np.isfinite(sd)) or sd <= 0:
        sd = 1.0
    return (r / sd).rename(f"{resid.name}_std"), sd

In [ ]:
def pick_macro_exog_cols(df: pd.DataFrame) -> list[str]:
    keep_prefix = ("RET_DRV_","CHG_DRV_","RET_POL_","CHG_POL_","RET_CN_","CHG_CN_",
                   "RET_US_","CHG_US_","RET_OTH_","CHG_OTH_","FEAT_",
                   "RET_CNY_SPOT","RET_CNH_SPOT")
    cols = []
    for c in df.columns.astype(str):
        if c.startswith("CHG_CNH_") or (c.startswith("RET_CNH_") and c!="RET_CNH_SPOT"): continue
        if c.startswith("CHG_CNY_") or (c.startswith("RET_CNY_") and c!="RET_CNY_SPOT"): continue
        if c.startswith(keep_prefix): cols.append(c)
    out, seen = [], set()
    for c in cols:
        if c in df.columns and c not in seen:
            out.append(c); seen.add(c)
    return out

def clean_exog(Xe: pd.DataFrame, min_non_nan_frac: float = 0.90):
    Xe = Xe.replace([np.inf, -np.inf], np.nan).copy()
    for c in Xe.columns:
        Xe[c] = pd.to_numeric(Xe[c], errors="coerce")

    keep = Xe.notna().mean(axis=0) >= float(min_non_nan_frac)
    Xe = Xe.loc[:, keep]

    retchg = [c for c in Xe.columns if str(c).startswith(("RET_", "CHG_"))]
    other  = [c for c in Xe.columns if c not in retchg]

    if other:
        Xe[other] = Xe[other].ffill()       
    if retchg:
        Xe[retchg] = Xe[retchg].fillna(0.0) 

    nunq = Xe.nunique(dropna=True)
    Xe = Xe.loc[:, nunq > 1]
    return Xe

In [ ]:
def harx_design(r: pd.Series, X_exog: pd.DataFrame | None, rv_window: int):
    r = _as_clean_series(r).astype(float)
    r2 = r**2
    X_core = pd.DataFrame({
        "log_rv_d": np.log(r2.clip(EPS)),
        "log_rv_w": np.log(r2.rolling(5, min_periods=5).mean().clip(EPS)),
        "log_rv_m": np.log(r2.rolling(22,min_periods=22).mean().clip(EPS)),
    }, index=r.index)
    y_next = logrv_proxy(r, rv_window).shift(-1).rename("y_next")
    X = X_core
    if X_exog is not None:
        X = pd.concat([X, clean_exog(X_exog.reindex(r.index))], axis=1)
    df = pd.concat([y_next, X], axis=1).replace([np.inf,-np.inf], np.nan).dropna()
    return df.drop(columns=["y_next"]), df["y_next"]

def rolling_harx_with_attrib(r, X_exog, oos_start, rv_window=5, lookback=1260, refit_every=21, min_fit_rows=400):
    r = _as_clean_series(r)
    X_all, y_all = harx_design(r, X_exog, rv_window)
    yhat_next = pd.Series(index=y_all.index, dtype=float)
    contrib   = pd.DataFrame(index=y_all.index, columns=list(X_all.columns), dtype=float)

    idx = r.index
    start_i = idx.get_indexer([oos_start], method="bfill")[0]
    i = start_i
    while i < len(idx):
        t_date = idx[i]
        end_fit_date = idx[i-1] if i>0 else t_date
        j_date = idx[min(len(idx)-1, i+refit_every-1)]

        df_fit = pd.concat([y_all, X_all], axis=1).loc[:end_fit_date].dropna()
        if len(df_fit) < min_fit_rows:
            i += refit_every; continue
        df_fit = df_fit.iloc[-lookback:] if len(df_fit) > lookback else df_fit

        Y = df_fit.iloc[:,0]
        Xmat = df_fit.iloc[:,1:].dropna()
        Y = Y.reindex(Xmat.index)
        if len(Xmat) < 250:
            i += refit_every; continue

        fit = sm.OLS(Y.values, sm.add_constant(Xmat, has_constant="add").values).fit()
        beta = pd.Series(fit.params, index=["const"] + list(Xmat.columns)).fillna(0.0)

        df_pred = X_all.loc[t_date:j_date].dropna()
        if len(df_pred) > 0:
            yhat_next.loc[df_pred.index] = fit.predict(sm.add_constant(df_pred, has_constant="add").values)
            b = beta.reindex(["const"] + list(df_pred.columns)).fillna(0.0)
            contrib.loc[df_pred.index, df_pred.columns] = df_pred.mul(b[df_pred.columns], axis=1).values

        i += refit_every

    yhat_next.name = f"HARX_logRV_next_w{rv_window}"
    return yhat_next, contrib

def winsor_df(df, qlo=0.01, qhi=0.99):
    out = df.copy()
    for c in out.columns:
        s = out[c].astype(float)
        lo, hi = float(s.quantile(qlo)), float(s.quantile(qhi))
        out[c] = s.clip(lo, hi)
    return out

def harx_monthly_attrib(r_std, X_exog, oos_start, window_start="2020-01-01",
                        rv_window=5, top_k=6, winsor_q=(0.01,0.99)):
    yhat_next, contrib = rolling_harx_with_attrib(r_std, X_exog, oos_start, rv_window=rv_window)
    rv_real = rv_proxy(r_std, rv_window).rename("RV_real")
    rv_hat  = np.exp(yhat_next).shift(1).rename("RV_hat")  
    top = pd.concat([rv_real, rv_hat], axis=1).dropna()
    top = top.loc[top.index >= pd.Timestamp(window_start)]

    # macro-only contributions (exclude HAR core)
    contrib2 = contrib.reindex(top.index).fillna(0.0)
    har_core = {"log_rv_d","log_rv_w","log_rv_m"}
    contrib2 = contrib2[[c for c in contrib2.columns if c not in har_core]]
    contrib2 = winsor_df(contrib2, winsor_q[0], winsor_q[1])

    Cm = contrib2.resample("M").mean()
    score = Cm.abs().mean().sort_values(ascending=False)
    keep = list(score.head(top_k).index)
    Cm_small = Cm[keep].copy()
    Cm_small["Other"] = Cm.drop(columns=keep).sum(axis=1)

    # metrics (OOS)
    oos_mask = top.index >= pd.Timestamp(oos_start)
    met = {
        "corr_logRV_oos": corr_logrv(top.loc[oos_mask,"RV_real"], top.loc[oos_mask,"RV_hat"]),
        "qlike_oos":      qlike(top.loc[oos_mask,"RV_real"], top.loc[oos_mask,"RV_hat"])
    }
    return top, Cm_small, met


In [56]:
# --- A1) infer a date from doc_id like news20240115 / news_2024-01-15 / etc ---
def infer_date_from_doc_id(doc_id: str):
    s = str(doc_id)
    m = re.search(r"(20\d{2})[-_/]?(0[1-9]|1[0-2])[-_/]?([0-2]\d|3[01])", s)
    if not m:
        return pd.NaT
    return pd.Timestamp(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")

sent_daily = df_finbert.copy()
sent_daily["news_date"] = sent_daily["doc_id"].apply(infer_date_from_doc_id)

# fallback: use file mtime if doc_id has no date
if sent_daily["news_date"].isna().any():
    mtime = {f.stem: pd.Timestamp(f.stat().st_mtime, unit="s").normalize() for f in clean_files}
    sent_daily.loc[sent_daily["news_date"].isna(), "news_date"] = sent_daily.loc[
        sent_daily["news_date"].isna(), "doc_id"
    ].map(mtime)

sent_daily["sent_signal"] = (sent_daily["sent_score"].abs() > 0.30).astype(int)
sent_daily = sent_daily.dropna(subset=["news_date"]).set_index("news_date").sort_index()

display(sent_daily[["doc_id","label","confidence","sent_score","sent_signal","positive","negative","neutral"]])

,doc_id,label,confidence,sent_score,sent_signal,positive,negative,neutral
news_date,,,,,,,,
2026-01-11,news1_1202,negative,0.850414,-0.748191,1,0.102223,0.850414,0.047364
2026-01-11,news2_1027,positive,0.499244,0.030796,0,0.499244,0.468448,0.032309
2026-01-11,news3_1223,positive,0.753010,0.684305,1,0.753010,0.068705,0.178285
2026-01-11,news4_1223,negative,0.941993,-0.913367,1,0.028626,0.941993,0.029382
2026-01-11,news5_1028,negative,0.948774,-0.913763,1,0.035011,0.948774,0.016215


In [ ]:

DATA_PATH = Path("FINAL_Data.csv")
IS_END    = pd.Timestamp("2019-12-31")
OOS_START = pd.Timestamp("2020-01-01")
TENORS    = ["1M","3M","6M","1Y"]
SMILES    = ["ATM","25DRR","25DBF","10DRR","10DBF"]
EPS = 1e-12

# ------------------------
# 0) Load panel0 from FINAL_Data.csv (same taxonomy)
# ------------------------
def parse_mixed_date(x):
    dt = pd.to_datetime(x, errors="coerce")
    if not pd.isna(dt):
        return dt
    try:
        xf = float(x)
        if 30000 < xf < 50000:
            return pd.to_datetime("1899-12-30") + pd.to_timedelta(xf, unit="D")
    except Exception:
        pass
    return pd.NaT

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df_raw["Date_parsed"] = df_raw["Date"].apply(parse_mixed_date)

df = (
    df_raw.dropna(subset=["Date_parsed"])
          .drop(columns=["Date"])
          .rename(columns={"Date_parsed": "Date"})
          .sort_values("Date")
          .set_index("Date")
)

# Surface nodes EXACTLY like your data: CNH_1M_ATM etc
SURFACE_COLS = [c for c in df.columns if re.match(r"^(CNY|CNH)_(1M|3M|6M|1Y)_(ATM|25DRR|25DBF|10DRR|10DBF)$", str(c))]

SPOT_COLS = [c for c in [
    "CNY_SPOT","CNH_SPOT",
    "CNY_CNY_PBOC_FIXING","CNH_CNH_PBOC_FIXING",
    "FEAT_CNY_Spot_Fix_Dev"
] if c in df.columns]

DRIVER_LEVEL_COLS = [c for c in df.columns if str(c).startswith(("DRV_","POL_","CN_","US_","OTH_"))]
RETURN_COLS       = [c for c in df.columns if str(c).startswith("RET_")]
CHANGE_COLS       = [c for c in df.columns if str(c).startswith("CHG_")]

panel = df[SURFACE_COLS + SPOT_COLS + DRIVER_LEVEL_COLS + RETURN_COLS + CHANGE_COLS].copy()
panel = panel.dropna(subset=SURFACE_COLS, how="all")                # drop days with zero surface
panel = panel.apply(pd.to_numeric, errors="coerce").sort_index()
panel0 = panel.copy()

print("Frozen panel0:", panel0.shape, panel0.index.min().date(), "->", panel0.index.max().date())
print("Surface cols:", len(SURFACE_COLS))

# ------------------------
# 1) Surface block helpers + CNH_START detection
# ------------------------
def surface_cols_for(df, mkt, tenor, smile):
    pat = re.compile(rf"^{mkt}_{tenor}_{smile}$")
    return [c for c in df.columns if pat.match(str(c))]

def surface_block_cols(df, mkt, smile):
    cols = []
    for t in TENORS:
        cols += surface_cols_for(df, mkt, t, smile)
    return cols

def first_valid_date(df, cols):
    if not cols:
        return None
    s = df[cols].notna().any(axis=1)
    return s.index[s.values][0] if s.any() else None

# blocks (CNY)
ATM_CNY  = surface_block_cols(panel0, "CNY", "ATM")
RR25_CNY = surface_block_cols(panel0, "CNY", "25DRR")
BF25_CNY = surface_block_cols(panel0, "CNY", "25DBF")
RR10_CNY = surface_block_cols(panel0, "CNY", "10DRR")
BF10_CNY = surface_block_cols(panel0, "CNY", "10DBF")

# blocks (CNH)
ATM_CNH  = surface_block_cols(panel0, "CNH", "ATM")
RR25_CNH = surface_block_cols(panel0, "CNH", "25DRR")
BF25_CNH = surface_block_cols(panel0, "CNH", "25DBF")
RR10_CNH = surface_block_cols(panel0, "CNH", "10DRR")
BF10_CNH = surface_block_cols(panel0, "CNH", "10DBF")

CNH_START = first_valid_date(panel0, ATM_CNH + RR25_CNH + BF25_CNH + RR10_CNH + BF10_CNH)
print("CNH_START:", None if CNH_START is None else CNH_START.date())


/var/folders/tl/hb3sh0p16wb8f6cdsnkjq_wm0000gn/T/ipykernel_52949/1313070588.py:18: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dt = pd.to_datetime(x, errors="coerce")


Frozen panel0: (5929, 145) 2004-01-03 -> 2025-12-23
Surface cols: 40
CNH_START: 2011-01-03


In [ ]:
def _try_join_pca(pf, panel0, df_is, cols, mode, n_components, prefix):
    if (cols is None) or (len(cols) == 0):
        print(f"[skip PCA] {prefix}: empty cols")
        return pf
    try:
        mdl = fit_pca_block(df_is, cols, mode=mode, n_components=n_components)
        return pf.join(apply_pca_block(panel0, mdl, prefix))
    except Exception as e:
        print(f"[skip PCA] {prefix}: {type(e).__name__}: {e}")
        return pf

def build_factors_safe(panel0: pd.DataFrame, is_end: pd.Timestamp):
    panel_is = panel0.loc[:is_end].copy()
    pf = panel0.copy()

    pf = _try_join_pca(pf, panel0, panel_is, ATM_CNY,  "dlog",   2, "CNY_ATM")
    pf = _try_join_pca(pf, panel0, panel_is, RR25_CNY, "dlevel", 1, "CNY_25RR")
    pf = _try_join_pca(pf, panel0, panel_is, BF25_CNY, "dlevel", 1, "CNY_25BF")
    pf = _try_join_pca(pf, panel0, panel_is, RR10_CNY, "dlevel", 1, "CNY_10RR")
    pf = _try_join_pca(pf, panel0, panel_is, BF10_CNY, "dlevel", 1, "CNY_10BF")

    if (CNH_START is not None) and (CNH_START <= is_end):
        panel_is_cnh = panel_is.loc[panel_is.index >= CNH_START].copy()
        pf = _try_join_pca(pf, panel0, panel_is_cnh, ATM_CNH,  "dlog",   2, "CNH_ATM")
        pf = _try_join_pca(pf, panel0, panel_is_cnh, RR25_CNH, "dlevel", 1, "CNH_25RR")
        pf = _try_join_pca(pf, panel0, panel_is_cnh, BF25_CNH, "dlevel", 1, "CNH_25BF")
        pf = _try_join_pca(pf, panel0, panel_is_cnh, RR10_CNH, "dlevel", 1, "CNH_10RR")
        pf = _try_join_pca(pf, panel0, panel_is_cnh, BF10_CNH, "dlevel", 1, "CNH_10BF")
    else:
        print("[skip CNH PCA] CNH_START is None or after IS_END.")

    FACTOR_COLS = [c for c in pf.columns if re.match(r"^(CNY|CNH)_(ATM|25RR|25BF|10RR|10BF)_PC\d+$", str(c))]
    return pf, FACTOR_COLS, CNH_START

panel_factors, FACTOR_COLS, CNH_START = build_factors_safe(panel0, IS_END)

if len(FACTOR_COLS) == 0:
    raise ValueError("No PCA factors were created. Check that SURFACE_COLS matched (CNY/CNH_*_*_*) and PCA fns exist.")

print("Factor cols:", len(FACTOR_COLS))


Factor cols: 12


In [71]:
def make_ar1_residuals(panel_factors: pd.DataFrame, factor_cols: list, is_end: pd.Timestamp) -> dict:
    resid_map = {}
    for c in factor_cols:
        s = panel_factors[c].dropna().astype(float)
        s_is = s.loc[:is_end]
        if len(s_is) < 300:
            continue
        y = s_is.iloc[1:].values
        X = sm.add_constant(s_is.iloc[:-1].values)
        fit = sm.OLS(y, X).fit()
        c0, phi = float(fit.params[0]), float(fit.params[1])
        e = (s - (c0 + phi * s.shift(1))).dropna()
        resid_map[c] = e.rename(f"{c}_resid_AR1")
    return resid_map

resid_map = make_ar1_residuals(panel_factors, FACTOR_COLS, IS_END)
if len(resid_map) == 0:
    raise ValueError("resid_map is empty (not enough IS rows or factors too sparse).")

FAC = "CNH_ATM_PC1" if "CNH_ATM_PC1" in resid_map else list(resid_map.keys())[0]
resid = resid_map[FAC].copy()

if (CNH_START is not None) and str(FAC).startswith("CNH_"):
    resid = resid.loc[resid.index >= CNH_START]

r_std, _ = standardise_resid(resid, IS_END)

print("FAC:", FAC)
print("r_std:", len(r_std), r_std.index.min().date(), "->", r_std.index.max().date())

FAC: CNH_ATM_PC1
r_std: 4050 2011-01-03 -> 2025-12-23


In [ ]:
import re
import pandas as pd
import numpy as np

def infer_date_from_doc_id(doc_id: str, fallback_year: int = 2025) -> pd.Timestamp:
    s = str(doc_id)
    # Make it more relevant for both
    # Case 1: has full YYYY-MM-DD somewhere
    m = re.search(r"(20\d{2})-(\d{2})-(\d{2})", s)
    if m:
        return pd.Timestamp(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")

    # Case 2: pattern like news1_1202 or news3_1223 (MMDD)
    m2 = re.search(r"(?:^|_)(\d{4})(?:$|_)", s)
    if m2:
        mmdd = m2.group(1)
        mm, dd = int(mmdd[:2]), int(mmdd[2:])
        if 1 <= mm <= 12 and 1 <= dd <= 31:
            return pd.Timestamp(f"{fallback_year}-{mm:02d}-{dd:02d}")

    return pd.NaT


sent_daily = df_finbert.copy()
sent_daily["news_date"] = sent_daily["doc_id"].apply(lambda x: infer_date_from_doc_id(x, fallback_year=2025))

if sent_daily["news_date"].isna().any():
    mtime = {f.stem: pd.Timestamp(f.stat().st_mtime, unit="s").normalize() for f in clean_files}
    miss = sent_daily["news_date"].isna()
    sent_daily.loc[miss, "news_date"] = sent_daily.loc[miss, "doc_id"].map(mtime)

sent_daily["sent_signal"] = (sent_daily["sent_score"].abs() > 0.30).astype(int)
sent_daily = sent_daily.dropna(subset=["news_date"]).set_index("news_date").sort_index()

print(sent_daily[["doc_id"]].head(10))
print("Unique news dates:", sent_daily.index.nunique(), "out of", len(sent_daily))
print("News date range:", sent_daily.index.min().date(), "->", sent_daily.index.max().date())

                doc_id
news_date             
2025-10-27  news2_1027
2025-10-28  news5_1028
2025-12-02  news1_1202
2025-12-23  news3_1223
2025-12-23  news4_1223
Unique news dates: 4 out of 5
News date range: 2025-10-27 -> 2025-12-23


In [77]:
exog_cols = pick_macro_exog_cols(panel_factors)
X_exog = clean_exog(panel_factors[exog_cols].copy(), min_non_nan_frac=0.90)

yhat_next, contrib = rolling_harx_with_attrib(
    r=r_std,
    X_exog=X_exog,
    oos_start=OOS_START,
    rv_window=5,
    lookback=1260,
    refit_every=21,
    min_fit_rows=400
)

# macro-only contrib (drop HAR core terms)
HAR_CORE = {"log_rv_d", "log_rv_w", "log_rv_m"}
contrib_macro = (
    contrib.drop(columns=[c for c in HAR_CORE if c in contrib.columns], errors="ignore")
          .replace([np.inf, -np.inf], np.nan)
          .fillna(0.0)
)


In [ ]:
def align_news_to_panel_date(news_dates: pd.DatetimeIndex, panel_index: pd.DatetimeIndex) -> pd.Series:
    tmp = pd.DataFrame({"news_date": pd.to_datetime(news_dates)}).sort_values("news_date")
    pan = pd.DataFrame({"panel_date": pd.to_datetime(panel_index).sort_values()})
    out = pd.merge_asof(tmp, pan, left_on="news_date", right_on="panel_date", direction="backward")
    return out.set_index("news_date")["panel_date"]

sent_daily2 = sent_daily.copy()
sent_daily2["panel_date"] = align_news_to_panel_date(sent_daily2.index, contrib_macro.index)

sent_daily2 = sent_daily2.dropna(subset=["panel_date"])
sent_daily2 = sent_daily2.loc[sent_daily2["panel_date"].isin(contrib_macro.index)].copy()


In [ ]:
def generate_risk_alert(report_payload: dict, model: str = "gpt-4.1"):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    user_prompt = (
        "Write the report using this data (do not add anything outside it):\n\n"
        + json.dumps(report_payload, ensure_ascii=False, indent=2)
    )

    payload = {
        "model": model,
        "instructions": SYSTEM_PROMPT,
        "input": user_prompt,
        "text": {"format": {"type": "text"}},
        "temperature": 0.2,
        "max_output_tokens": 900
    }

    r = requests.post(API_URL, headers=headers, json=payload, timeout=90)
    r.raise_for_status()
    data = r.json()

    # Common convenience field (when available)
    if "output_text" in data and data["output_text"]:
        return data["output_text"].strip()

    # Fallback: assemble from output items
    out = []
    for item in data.get("output", []):
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text"):
                out.append(c.get("text", ""))
    return "\n".join(out).strip()

In [79]:
TOPK = 5
rows = []

for news_dt, row in sent_daily2.iterrows():
    t = row["panel_date"]  

    c = contrib_macro.loc[t].astype(float)
    c = c.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    top_idx = c.abs().sort_values(ascending=False).head(TOPK).index
    top = c.reindex(top_idx)

    out = {
        "news_date": news_dt,
        "panel_date": t,
        "doc_id": row["doc_id"],
        "label": row["label"],
        "confidence": float(row["confidence"]),
        "sent_score": float(row["sent_score"]),
        "sent_signal": int(row["sent_signal"]),
        "factor": FAC,
        "sum_macro_contrib": float(c.sum()),
    }
    for i, (k, v) in enumerate(top.items(), 1):
        out[f"driver_{i}"]  = k
        out[f"contrib_{i}"] = float(v)

    rows.append(out)

news_top5 = pd.DataFrame(rows).sort_values(["news_date"]).reset_index(drop=True)

display(news_top5)

,news_date,panel_date,doc_id,label,confidence,sent_score,sent_signal,factor,sum_macro_contrib,driver_1,contrib_1,driver_2,contrib_2,driver_3,contrib_3,driver_4,contrib_4,driver_5,contrib_5
0,2025-10-27,2025-10-27,news2_1027,positive,0.499244,0.030796,0,CNH_ATM_PC1,0.105335,CHG_POL_CN_REPO_1D_ONSHORE_bps,0.095025,CHG_POL_CN_REPO_7D_ONSHORE_bps,-0.079024,RET_DRV_CSI_300,0.045337,CHG_DRV_CN_2Y_YLD_bps,0.038733,CHG_DRV_CN_10Y_YLD_bps,-0.027541
1,2025-10-28,2025-10-28,news5_1028,negative,0.948774,-0.913763,1,CNH_ATM_PC1,0.039502,RET_DRV_BRENT,0.025549,CHG_DRV_CN_2Y_YLD_bps,0.020856,RET_DRV_CSI_300,-0.019637,RET_CNY_SPOT,0.011818,RET_DRV_DXY_US_DOLLAR_INDEX,-0.010813
2,2025-12-02,2025-12-02,news1_1202,negative,0.850414,-0.748191,1,CNH_ATM_PC1,0.073138,CHG_POL_CN_REPO_1D_ONSHORE_bps,-0.080114,RET_OTH_SGD,0.056127,CHG_DRV_CN_2Y_YLD_bps,0.039249,CHG_POL_CN_REPO_7D_ONSHORE_bps,0.036697,RET_DRV_BRENT,0.032353
3,2025-12-23,2025-12-22,news3_1223,positive,0.753010,0.684305,1,CNH_ATM_PC1,0.159928,CHG_DRV_CN_2Y_YLD_bps,0.094422,RET_DRV_BRENT,-0.035315,CHG_DRV_VIX_bps,0.028043,RET_DRV_CSI_300,0.023260,RET_OTH_SGD,0.022838
4,2025-12-23,2025-12-22,news4_1223,negative,0.941993,-0.913367,1,CNH_ATM_PC1,0.159928,CHG_DRV_CN_2Y_YLD_bps,0.094422,RET_DRV_BRENT,-0.035315,CHG_DRV_VIX_bps,0.028043,RET_DRV_CSI_300,0.023260,RET_OTH_SGD,0.022838


# Implementation of the reporting system 

In [ ]:
from pathlib import Path
import re
import pandas as pd

# CONFIG
OUTPUTS_DIR = Path("NLP Outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DIR = Path("NLP Data Cleaned")  # cleaned news*.txt live here

def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-\.]+", "_", s)

def load_cleaned_news_text(doc_id: str) -> str:
    p = CLEAN_DIR / f"{doc_id}.txt"
    if p.exists():
        return p.read_text(encoding="utf-8", errors="ignore").strip()
    hits = list(CLEAN_DIR.glob(f"{doc_id}*.txt"))
    if hits:
        return hits[0].read_text(encoding="utf-8", errors="ignore").strip()
    return "N/A"

def build_payload_from_row(row: pd.Series, news_text: str) -> dict:
    drivers = []
    for k in range(1, 6):
        d = row.get(f"driver_{k}")
        c = row.get(f"contrib_{k}")
        if pd.notna(d) and pd.notna(c):
            drivers.append({"driver": str(d), "contribution_to_pred_logRV": float(c)})

    return {
        "as_of_date": str(pd.to_datetime(row["panel_date"]).date()),
        "pair": "USD/CNH",
        "market_regime_inputs": {
            "skew_zscore": row.get("skew_zscore", "N/A"),
            "basis_spread_bps": row.get("basis_spread_bps", "N/A"),
            "sum_macro_contrib": float(row.get("sum_macro_contrib")) if pd.notna(row.get("sum_macro_contrib")) else "N/A",
        },
        "iv_factor": {"factor_name": str(row.get("factor", "N/A"))},
        "sentiment": {
            "label": str(row.get("label", "N/A")),
            "confidence": float(row.get("confidence")) if pd.notna(row.get("confidence")) else "N/A",
            "sent_score": float(row.get("sent_score")) if pd.notna(row.get("sent_score")) else "N/A",
            "sent_signal": int(row.get("sent_signal")) if pd.notna(row.get("sent_signal")) else "N/A",
        },
        "harx_macro_drivers_top5": drivers,
        "news_cleaned_text": news_text if news_text else "N/A",
        "meta": {
            "doc_id": str(row.get("doc_id", "N/A")),
            "news_date": str(pd.to_datetime(row["news_date"]).date()) if pd.notna(row.get("news_date")) else "N/A",
        }
    }

# RUN FOR 4 UNIQUE PANEL DAYS (from news_top5)
if "news_top5" not in globals():
    raise ValueError("news_top5 not found. Run your TOPK table construction cell first.")

df = news_top5.copy()

df["news_date"]  = pd.to_datetime(df["news_date"], errors="coerce")
df["panel_date"] = pd.to_datetime(df["panel_date"], errors="coerce")

# pick first 4 unique panel days (sorted)
unique_days = sorted([pd.Timestamp(d).normalize() for d in df["panel_date"].dropna().unique()])[:4]
if not unique_days:
    raise ValueError("No valid panel_date values in news_top5.")

print("Unique panel days to run:", [str(d.date()) for d in unique_days])

for day in unique_days:
    day_df = df[df["panel_date"].dt.normalize() == day].copy()
    if day_df.empty:
        continue

    reports = []
    for _, r in day_df.iterrows():
        doc_id = str(r["doc_id"])
        news_text = load_cleaned_news_text(doc_id)
        payload = build_payload_from_row(r, news_text)

        txt = generate_risk_alert(payload, model="gpt-4.1") 
        header = (
            f"\n\n{'='*90}\n"
            f"DOC: {doc_id} | news_date={pd.to_datetime(r['news_date']).date()} | panel_date={pd.to_datetime(r['panel_date']).date()}\n"
            f"{'='*90}\n"
        )
        reports.append(header + txt)

    out_text = f"UBS Professional Risk Alerts — Panel Date: {day.date()}\n" + "\n".join(reports)
    out_file = OUTPUTS_DIR / f"risk_alert_{safe_filename(str(day.date()))}.txt"
    out_file.write_text(out_text, encoding="utf-8")
    print(f"✅ saved: {out_file}")

print(f"\nDone. Outputs in: {OUTPUTS_DIR.resolve()}")

Unique panel days to run: ['2025-10-27', '2025-10-28', '2025-12-02', '2025-12-22']
✅ saved: NLP Outputs/risk_alert_2025-10-27.txt
✅ saved: NLP Outputs/risk_alert_2025-10-28.txt
✅ saved: NLP Outputs/risk_alert_2025-12-02.txt
✅ saved: NLP Outputs/risk_alert_2025-12-22.txt

Done. Outputs in: /Users/nigelli/Desktop/Columbia MAFN/UBS FINAI/Code/NLP Outputs
